In [1]:
import pandas as pd
from datetime import datetime, time, timedelta
import tqdm

# Machine Parameters

In [2]:
shift1_start = time(7, 0, 1)
shift1_end   = time(15, 0, 0)
shift2_start = time(15, 0, 1)
shift2_end   = time(23, 0, 0)
def assign_shift(t):
    if shift1_start <= t <= shift1_end:
        return 'Shift 1'
    elif shift2_start <= t <= shift2_end:
        return 'Shift 2'
    else:
        return 'Shift 3'

In [3]:
def preprocess_columns(df_input):
    df = df_input.copy()
    df = df.drop(columns=['Time'])
    df.columns = df.columns.str.strip()
    df.insert(0, 'Creping', (df['Yankee Speed'] - df['Pope Reel Speed']) * 100 / df['Yankee Speed'])
    df.insert(0, 'Time', df_input['Time'].apply(lambda x: datetime.strptime(x.split(' ')[1], '%H:%M:%S').time()))
    df['Shift'] = df['Time'].apply(assign_shift)


    
    df.insert(0, 'Date', df_input['Time'].apply(lambda x: x.split(' ')[0]))
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y')
    #Assign date shift 3 to the previous day (because shift 3 is from 11 PM prev day to 7 AM next day,
    #but it should be assigned to the day when it starts, not the day when it ends)
    #Index is used because we need to access the date value and shift value at once.
    df['Date'] = [df['Date'][index] - timedelta(days=1) 
                if
                df['Shift'][index] == 'Shift 3' 
                else df['Date'][index] 
                for index in range(len(df['Shift']))]

    
    return df

In [4]:
raw17 = pd.read_csv('17-130326.csv', delimiter = ';', encoding = 'utf-16').dropna(inplace = False).reset_index(drop = True)
raw17.rename(columns={'Speed Yankee': 'Yankee Speed', 'Preasure Yankee': 'Yankee Pressure'}, inplace=True)


In [5]:
pm_17 = preprocess_columns(raw17)

In [6]:
pm_17['Datetime'] = pm_17.apply(lambda row: datetime.combine(row['Date'], row['Time']), axis=1)
pm_17['Datetime']

0      2026-03-11 14:02:21
1      2026-03-11 14:03:21
2      2026-03-11 14:04:21
3      2026-03-11 14:05:21
4      2026-03-11 14:06:21
               ...        
2565   2026-03-13 08:47:21
2566   2026-03-13 08:48:21
2567   2026-03-13 08:49:21
2568   2026-03-13 08:50:21
2569   2026-03-13 08:51:21
Name: Datetime, Length: 2570, dtype: datetime64[us]

## 0th join, to obtain the average parameters for each jumbo

Function to convert time input reel time to hh:mm format.

In [50]:
import pandas as pd
from datetime import datetime
import math

def time_to_hms(val):
    s = str(val).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return pd.NA

    # If already contains colon, parse parts directly
    if ':' in s:
        parts = s.split(':')
        h = int(parts[0])
        m = int(parts[1]) if len(parts) > 1 and parts[1] != '' else 0
        sec = int(parts[2]) if len(parts) > 2 and parts[2] != '' else 0

    # If contains dot, treat left as hours and right as minutes (common human shorthand)
    elif '.' in s:
        left, right = s.split('.', 1)
        if int(left) >= 24:
            return pd.NA  # Invalid hour value
        left = 0 if left == "24" else left  # Handle "24" as "00"
        h = int(left) if left != '' else 0

        # If right part is short (1 or 2 digits) treat it as minutes (e.g., "4.1" -> 4:01, "11.55" -> 11:55)
        if len(right) <= 2:
            m = int(right)
            sec = 0
        else:
            # If right part is longer, treat the whole value as a decimal hour (fallback)
            # e.g., "4.125" -> 4.125 hours -> convert fractional hour to minutes
            f = float(s)
            total_minutes = int(round((f - math.floor(f)) * 60))
            m = total_minutes
            sec = 0

    # No separator: treat as hours only (e.g., "6" -> 06:00:00)
    else:
        h = int(float(s))
        m = 0
        sec = 0

    # Normalize minutes >= 60 into hours
    if m >= 60:
        extra_h = m // 60
        h = (h + extra_h) % 24
        m = m % 60

    return f"{h:02d}:{m:02d}:{sec:02d}"



In [57]:
reel_pm17 = pd.read_excel("D:/mike_doc_sun/proyek_predict_spec/MDS/DATA REEL JAN'26 SD APR'26 (PM12, PM15, PM17).xlsx", sheet_name = 'PM17')
reel_pm17['Time'] = reel_pm17['Time'].apply(time_to_hms)
reel_pm17['Tanggal'] = pd.to_datetime(reel_pm17['Tanggal'], format='%d.%m.%y')

reel_pm17 = reel_pm17.dropna(subset = ['Time']).reset_index(drop = True)
reel_pm17['Timestamp'] = reel_pm17['Tanggal'].astype(str) + ' ' + reel_pm17['Time'].astype(str)
reel_pm17['Timestamp'] = pd.to_datetime(reel_pm17['Timestamp'], format='%Y-%m-%d %H:%M:%S')

In [58]:
reel_pm17['Timestamp']

0      2026-01-01 07:15:00
1      2026-01-01 08:05:00
2      2026-01-01 08:55:00
3      2026-01-01 09:35:00
4      2026-01-01 10:15:00
               ...        
1986   2026-04-07 02:05:00
1987   2026-04-07 03:03:00
1988   2026-04-07 04:15:00
1989   2026-04-07 05:15:00
1990   2026-04-07 06:03:00
Name: Timestamp, Length: 1991, dtype: datetime64[us]

In [49]:
reel_pm17.isna().sum()

Time                       2
Tanggal                    0
Grade                    101
Shift                      0
Reel                       0
Bw                         0
Thickness                  0
MDT                        0
CDT                        0
MDWT                       0
RATIO MDWT (MDWT/MDT)     28
MDS                        0
Brightness                 0
Insp. Status               0
Timestamp                  2
dtype: int64

In [ ]:
reel

## 1st Join, to obtain the CSF & moisture for every pulp name

In [6]:
#Join key = pulp_csv_table["Jenis pulp"]-> inner join with pulp_qc_notes_table["Note"]
pulp_csf_table = pd.read_excel('D:/mike_doc_sun/proyek_predict_spec/MDS/QC - Pulp.xlsx', sheet_name = "Sheet1")
pulp_csf_table = clean_column_names(pulp_csf_table)
pulp_csf_table['Jenis pulp'] = pulp_csf_table['Jenis pulp'].str.strip()
pulp_csf_table.isnull().sum()


Vendor pulp      0
Jenis pulp       0
CSF              0
Moisture pulp    0
dtype: int64

In [7]:
#Join key = pulp_csv_table["Jenis pulp"]-> inner join with pulp_qc_notes_table["Note"]
pulp_qc_notes_table = pd.read_excel('D:/mike_doc_sun/proyek_predict_spec/MDS/notes pulp.xlsx', sheet_name = "Sheet1")
pulp_qc_notes_table = clean_column_names(pulp_qc_notes_table)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.strip()
pulp_qc_notes_table['Batch Pulp'] = pulp_qc_notes_table['Batch Pulp'].str.strip()
pulp_qc_notes_table.dropna(inplace = True)
pulp_qc_notes_table.isnull().sum()

Material Description    0
Batch Description       0
Nama Pulp               0
Batch Pulp              0
Note                    0
dtype: int64

In [8]:
#Standardizing name
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED HARDWOOD KRAFT PULP- ACACIA PRIME', 'BLEACHED HARDWOOD KRAFT PULP - ACACIA PRIME', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED HARDWOOD KRAFT PULP ACACIA PRIME', 'BLEACHED HARDWOOD KRAFT PULP - ACACIA PRIME', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED HARDWOOD KRAFT PULP ACACAI PRIME', 'BLEACHED HARDWOOD KRAFT PULP - ACACIA PRIME', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED HARDWOOD KRAFT PULP -  ACACIA PRIME', 'BLEACHED HARDWOOD KRAFT PULP - ACACIA PRIME', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED HARDWOOD KRAFT PULP-ACACIA PRIME', 'BLEACHED HARDWOOD KRAFT PULP - ACACIA PRIME', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED SOFTWOOD KRAFT PULP EX ASPA - FSC MIX CREDIT', 'BLEACHED SOFTWOOD KRAFT PULP EX. ASPA FSC MIX CREDIT', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED SOFTWOOD KRAFT PULP EX ASPA - FSC MIX', 'BLEACHED SOFTWOOD KRAFT PULP EX. ASPA FSC MIX CREDIT', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED SOFTWOOD KRAFT PULP EX ASPA MILL, SWEDEN - FSC MIX CREDIT', 'BLEACHED SOFTWOOD KRAFT PULP EX. ASPA FSC MIX CREDIT', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED SOFTWOOD KRAFT PULP EX ASPA FSC MIX CREDIT', 'BLEACHED SOFTWOOD KRAFT PULP EX. ASPA FSC MIX CREDIT', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED SOFTWOOD KRAFT PULP EX ASPA- FSC MIX CREDIT', 'BLEACHED SOFTWOOD KRAFT PULP EX. ASPA FSC MIX CREDIT', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('BLEACHED SOFTWOOD KRAFT PULP-CONIFER EX UPM MIL, FINLAND- FSC CONTROLL', 'UPM CONIFER (BLEACHED SOFTWOOD KRAFT PULP) - FSC', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('WOODPULP BLEACHED SULPHATE PULP (CANFOR FSC CONTROLLED WOOD)', 'WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFOR FSC CONTROLLED WOOD', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('WOODPULP BLEACHED SOFTWOOD SULPHATE PULP (CANFOR FSC CONTROLLED WOOD)', 'WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFOR FSC CONTROLLED WOOD', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('WOODPULP BLEACHED SOFTWOOD SULPHATE PULP', 'WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFOR FSC CONTROLLED WOOD', case=False)
pulp_qc_notes_table['Note'] = pulp_qc_notes_table['Note'].str.replace('WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFOR FSC CONTROLLED WOOD CANFOR FSC CONTROLLED WOOD', 'WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFOR FSC CONTROLLED WOOD', case=False)

In [9]:
pulp_csf_table['Jenis pulp'].unique().tolist()

['BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC',
 'BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS OFF GRADE A1 PEFC',
 'BLEACHED EUCALYPTUS KRAFT PULP FSC MIX GRADE',
 'BLEACHED EUCALYPTUS KRAFT PULP',
 'BLEACHED EUCALYPTUS KRAFT PULP FSC MIX CREDIT',
 'BLEACHED HARDWOOD KRAFT PULP - ACACIA PRIME',
 'UPM BLEACHED EUCALYPTUS KRAFT PULP FSC',
 'NORTHERN BLEACHED KRAFT PULP SW SCA PURE ECF',
 'UPM BETULA - BLEACHED EUCALYPTUS KRAFT PULP FSC MIX CREDIT',
 'NORTHERN BLEACHED KRAFT PULP SW SCA PURE ECF 90 - FSC MIX CREDIT',
 'NBSK SODRA BLUE - FSC MIX CREDIT SOFTWOOD PULP',
 'METSA PINE BLEACHED SOFTWOOD KRAFT PULP - FSC CONTROLLED WOOD',
 'UNBLEACHED SOFTWOOD KRAFT PULP EX. PROVENCE - FSC CONTOLLED WOOD',
 'BLEACHED SOFTWOOD KRAFT PULP PRIME',
 'BLEACHED SOFTWOOD KRAFT PULP OFF GRADE - FSC CONTROLLED WOOD',
 'BLEACHED EUCALYPTUS KRAFT PULP - GUAIBA B88 FSC MIX CREDIT',
 'UNBLEACHED KRAFT PULP ARAUCO FSC MIX CREDIT',
 'BLEACHED SOFTWOOD KRAFT PULP MERCER CELGAR OFF GRADE - FSC CONTROLLED WOO

In [10]:
#Removing unclear names
pulp_qc_notes_table = pulp_qc_notes_table[pulp_qc_notes_table['Note'] != 'KANDUNGAN PITCH SEDIKIT DENGAN UKURAN KECIL']
pulp_qc_notes_table = pulp_qc_notes_table[pulp_qc_notes_table['Note'] != 'KANDUNGAN PITCH SEDANG DENGAN UKURAN KECIL']
pulp_qc_notes_table = pulp_qc_notes_table[pulp_qc_notes_table['Note'] != 'KANDUNGAN PITCH BANYAK SEKALI DENGAN UKURAN KECIL ']
pulp_qc_notes_table = pulp_qc_notes_table[pulp_qc_notes_table['Note'] != 'KANDUNGAN PITCH BANYAK SEKALI DENGAN UKURAN KECIL']

In [11]:
csf_per_batch = pd.merge(pulp_csf_table, pulp_qc_notes_table, how='inner', left_on='Jenis pulp', right_on='Note', suffixes=('_csf', '_qc'))
csf_per_batch_unincluded = pd.merge(pulp_csf_table, pulp_qc_notes_table, how='right_anti', left_on='Jenis pulp', right_on='Note')

In [12]:
len(csf_per_batch_unincluded['Note'].unique().tolist())

151

In [13]:
csf_per_batch_unincluded['Note'].unique().tolist()

['BLEACHED HARDWOOD KRAFT PULP - ACACIA PEFC 100%',
 'WET LBKP PULP',
 'BLEACHED HARDWOOD KRAFT PULP - ACACIA',
 'LBKP BLEACHED HARDWOOD KRAFT PULP - ACACIA PRIME EX INTIGUNA',
 'BLEACHED HARDWOOD KRAFT PULP ACACIA - APP PRIME',
 'BLEACHED XTRA PRIME ACACIA - BPT',
 'BLEACHED HARDWOOD KRAFT PULP - BPT',
 'BLEACHED HARDWOOD KRAFT PULP - BEST PULP FOR TISSUE (BPT)',
 'BLEACHED HARDWOOD KRAFT PULP- ACACIA',
 'PULP LBKP AVALAN ASURANSI',
 'LBKP',
 'BLEACHED HARDWOOD KRAFT PULP ACACIA',
 'PULP LBKP AVALAN EX ASURANSI',
 'BLEACHED HARDWOOD KRAFT PULP',
 'BLEACHED SOFTWOOD KRAFT PULP EX ASPA (ESSWELL)',
 'NBSK BILLERUDKORSNAS - FSC MIX CREDIT SOFTWOOD PULP',
 'NBSK BILLERUDKORNAS-FSC MIX CREDIT SOFTWOOD PULP',
 'BLEACHED HARDWOOD KRAFT PULP ACACIA ECF 100% PEFC CERTIFIED',
 'LBKP BLEACHED HARDWOOD KRAFT PULP ACACIA IK P EX CAKRAWALA',
 'BLEACHED HARDWOOD KRAFT PULP ACACIA ECF',
 'BLEACHED HARDWOOD KRAFT PULP - ACACIA APP PRIME',
 'BLEACHED HARDWOOD KRAFT PULP ACACIA APP PRIME',
 'CANFOR NBSK 

In [42]:
csf_per_batch

,Vendor pulp,Jenis pulp,CSF,Moisture pulp,Material Description,Batch Description,Nama Pulp,Batch Pulp,Note
0,GRAHA HIJAU NUSANTARA,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC,615,0.1050,EBKP Non FSC Lokal,GRAHA HIJAU OFF GRADE 252611,GRAHA HIJAU OFF GRADE,680OG25047,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC
1,GRAHA HIJAU NUSANTARA,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC,615,0.1050,EBKP Non FSC Lokal,GRAHA HIJAU OFF GRADE 251112,GRAHA HIJAU OFF GRADE,680OG25048,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC
2,GRAHA HIJAU NUSANTARA,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC,615,0.1050,EBKP Non FSC Lokal,GRAHA HIJAU OFF GRADE 253012,GRAHA HIJAU OFF GRADE,680OG26001,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC
3,GRAHA HIJAU NUSANTARA,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC,615,0.1050,EBKP Non FSC Lokal,GRAHA HIJAU OFF GRADE 253112,GRAHA HIJAU OFF GRADE,680OG26002,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS A1 PEFC
4,GRAHA HIJAU NUSANTARA,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS OFF ...,595,0.1180,EBKP Non FSC Lokal,GRAHA HIJAU OFF GRADE 250207,GRAHA HIJAU OFF GRADE,680OG25021,BLEACHED HARDWOOD KRAFT PULP - EUCALYPTUS OFF ...
...,...,...,...,...,...,...,...,...,...
300,CANFOR PULP LTD,WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFO...,710,0.1381,NBKP FSC Impor,CANFOR PRIME GRADE 222503,CANFOR PRIME GRADE,130PG22005,WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFO...
301,CANFOR PULP LTD,WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFO...,710,0.1381,NBKP FSC Impor,CANFOR PRIME GRADE 222403,CANFOR PRIME GRADE,130PG22003,WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFO...
302,CANFOR PULP LTD,WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFO...,710,0.1381,NBKP FSC Impor,CANFOR PRIME GRADE 243108,CANFOR PRIME GRADE,130PG24002,WOODPULP BLEACHED SOFTWOOD SULPHATE PULP CANFO...
303,OJI FIBRE SOLUTIONS (NZ) LIMITED,UNBLEACHED SOFTWOOD KRAFT PULP EX.OJI - FSC CO...,745,0.1238,NUKP FSC Impor,OJI FIBRE PRIME GRADE 250305,OJI FIBRE PRIME GRADE,400PG25001,UNBLEACHED SOFTWOOD KRAFT PULP EX.OJI - FSC CO...


## 2nd join, get the pulp average price

In [33]:
pulp_price = pd.read_excel('D:/mike_doc_sun/proyek_predict_spec/MDS/Pulp Usage Record Data Till 23 FEB 2026.xlsx', sheet_name = "ZMM")
pulp_price = clean_column_names(pulp_price)
pulp_price['Batch'] = pulp_price['Batch'].str.strip()
pulp_price.isnull().sum()

Material                   0
Material Description       0
Batch                      0
Mat+Batch                  0
Conversion Kg/bal        839
Average Price              0
Batch Description         45
Desc+Batch                 0
Claim FSC                119
Purchase order           172
Posting Date             172
Material Document        172
Di input ke excell      1843
dtype: int64

## 3rd join, get the pulp usage

In [34]:
pulp_usage = pd.read_excel('D:/mike_doc_sun/proyek_predict_spec/MDS/Pulp Usage Record Data Till 23 FEB 2026.xlsx', sheet_name = "Raw")
pulp_usage = clean_column_names(pulp_usage)
pulp_usage['Batch'] = pulp_usage['Batch'].str.strip()
pulp_usage.dropna(subset=['Batch', 'USAGE (BALES)'], inplace=True)

pulp_usage.isnull().sum()

                            0
Year                        0
Month                       0
Date                        0
USAGE LOCATION              0
SHIFT                      49
PULP ITEM                   0
PULP TYPE                   9
USAGE (BALES)               0
CONVERSION                  0
USAGE (Kg)                  0
Remarks                     0
Price                       6
PO                         19
Batch                       0
Posting Date               19
Material Document          19
Material Desc               0
PLANNING\n(ITEM)           82
PLANNING\n(TYPE)          110
PLANNING \n(Remarks)       93
STATUS                      5
Notes                   22996
dtype: int64